# BigGAN

---
## 目的
高解像度・高品質な条件付き画像生成ができるBigGAN [1]を構築し，Spectral Normalization (SN) やSelf-Attention，Class-conditional Batch Normalizationなど，これまでのノートブックで扱った要素技術を組み合わせることで大規模なGANを安定して学習させる工夫を理解する．

[1] Andrew Brock, Jeff Donahue and Karen Simonyan, "Large Scale GAN Training for High Fidelity Natural Image Synthesis," ICLR, 2019.\
[2] Han Zhang, Ian Goodfellow, Dimitris Metaxas and Augustus Odena, "Self-Attention Generative Adversarial Networks," ICML, 2019.\
[3] Takeru Miyato, Toshiki Kataoka, Masanori Koyama and Yuichi Yoshida, "Spectral Normalization for Generative Adversarial Networks," ICLR, 2018.</cell id="cell-0">


## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.parameter import Parameter
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## ネットワークの構築
BigGANは，膨大なマシンリソースを活用して高解像度な条件付き画像生成を実現した，cGANの一種です．Miyatoらが提案したSpectral Normalization GAN (SN-GAN) と，Zhangらが提案したSelf-Attention GAN (SA-GAN) をベースにネットワークを構成します．

### Spectral Normalization
GANの学習を安定させる手法としては，`wgan_gp.ipynb`で扱ったWGAN・WGAN-GPのように，Discriminator（Critic）が1-Lipschitz連続であることを保証する枠組みがよく使われます．しかし，WGANの重みクリッピングは強すぎる制約であり，WGAN-GPのGradient Penaltyも，勾配のノルムを計算するために2階微分が必要となり計算コストがかさむという課題がありました．

**Spectral Normalization (SN)** [3] は，各層の重み行列を，その最大特異値（Spectral Norm）で割ることで，層ごとに1-Lipschitz連続性を保証する手法です．誤差関数に正則化項を追加するのではなく重み自体を直接正規化するため，計算コストを抑えながら学習を安定させることができます．SN-GAN・SA-GANではDiscriminatorのみにSNを適用していましたが，BigGANでは**Generatorにも**SNを適用することで，より安定した学習を実現しています．</cell id="cell-3">


In [ ]:
def proj(x, y):
    return torch.mm(y, x.t()) * y / torch.mm(y, y.t())

def gram_schmidt(x, ys):
    for y in ys:
        x = x - proj(x, y)
    return x

def power_iteration(W, u_, update=True, eps=1e-12):
    us, vs, svs = [], [], []
    for i, u in enumerate(u_):
        with torch.no_grad():
            v = torch.matmul(u, W)
            v = F.normalize(gram_schmidt(v, vs), eps=eps)
            vs += [v]
            u = torch.matmul(v, W.t())
            u = F.normalize(gram_schmidt(u, us), eps=eps)
            us += [u]
            if update:
                u_[i][:] = u
        svs += [torch.squeeze(torch.matmul(torch.matmul(v, W.t()), u.t()))]
    return svs, us, vs

class SpectralNorm:
    def __init__(self, num_svs, num_itrs, num_outputs, transpose=False, eps=1e-12):
        self.num_itrs = num_itrs
        self.num_svs = num_svs
        self.transpose = transpose
        self.eps = eps
        for i in range(self.num_svs):
            self.register_buffer(f'u{i}', torch.randn(1, num_outputs))
            self.register_buffer(f'sv{i}', torch.ones(1))

    @property
    def u(self):
        return [getattr(self, f'u{i}') for i in range(self.num_svs)]

    @property
    def sv(self):
        return [getattr(self, f'sv{i}') for i in range(self.num_svs)]

    def W_(self):
        W_mat = self.weight.view(self.weight.size(0), -1)
        if self.transpose:
            W_mat = W_mat.t()
        for _ in range(self.num_itrs):
            svs, us, vs = power_iteration(W_mat, self.u, update=self.training, eps=self.eps)
        if self.training:
            with torch.no_grad():
                for i, sv in enumerate(svs):
                    self.sv[i][:] = sv
        return self.weight / svs[0]

### Self-Attention Layerの構築

![biggan_sa](images/biggan_sa.png)

SA-GAN [2] は，GANに初めてSelf-Attentionを導入した手法です．CIFAR-10やImageNetなどの自然画像は，姿勢や構図が一定でない（様々な見え方をとりうる）ため，畳み込み層のような近傍の画素しか参照できない局所的な演算だけでは，画像全体の構造の整合性を捉えきれず，崩れた画像が生成されやすいことが知られています．Generator・Discriminatorの特徴マップサイズが比較的大きい層にSelf-Attentionを挿入し，画像内の離れた位置の情報を直接参照できるようにすることで，この問題を緩和します．

Self-Attentionは，特徴マップに3種類の$1\times1$畳み込みを適用してquery・key・valueを作成し，通常のSelf-Attention（Transformerなどで用いられるものと同様）の計算方法でattention mapを作成します．</cell id="cell-5">


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.query_conv = SpectralNormConv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = SpectralNormConv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = SpectralNormConv2d(in_dim, in_dim, kernel_size=1)
        self.gamma = Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        b, c, w, h = x.size()
        query = self.query_conv(x).view(b, -1, w * h).permute(0, 2, 1)  # (B, Ch, W, H) -> (B, WxH, Ch)
        key = self.key_conv(x).view(b, -1, w * h)                       # (B, Ch, W, H) -> (B, Ch, WxH)
        attn = self.softmax(torch.bmm(query, key))                      # query @ key

        value = self.value_conv(x).view(b, -1, w * h)                   # (B, Ch, W, H) -> (B, Ch, WxH)
        out = torch.bmm(value, attn.permute(0, 2, 1))                   # value @ attn
        out = out.view(b, c, w, h)
        return self.gamma * out + x

### Conditional Batch Normalizationの構築
BigGANは，入力した潜在変数と条件からGenerator内の全層のBatch Normalizationのアフィンパラメータを生成します．
Pytorchなどで実装されているBatch Normalizationに少し手を加えるだけで実現できます．

In [ ]:
class ConditionalBatchNorm(nn.Module):
    def __init__(self, in_channel, n_condition=148):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channel, affine=False)

        self.embed = nn.Linear(n_condition, in_channel * 2)
        self.embed.weight.data[:, :in_channel] = 1
        self.embed.weight.data[:, in_channel:] = 0

    def forward(self, input, class_id):
        out = self.bn(input)
        embed = self.embed(class_id)
        gamma, beta = embed.chunk(2, 1)
        gamma = gamma.unsqueeze(2).unsqueeze(3)
        beta = beta.unsqueeze(2).unsqueeze(3)
        return gamma * out + beta

### Cross Replica Batch Normalizationについて
原論文のBigGANは，複数GPUによる分散学習を行っており，Generatorの最後のBatch Normalizationには，各GPU（replica）間で統計量（平均・分散）を同期する**Cross Replica Batch Normalization**を使用しています．本ノートブックは単一GPUでの推論のみを行うため，GPU間の同期は不要です．そのため，通常の`nn.BatchNorm2d`を使用します（学習済みモデルのパラメータ名はどちらの実装でも同一のため，`load_state_dict`で問題なく読み込めます）．</cell id="cell-9">


### ネットワーク構築に必要なモジュールの改造

In [ ]:
class SpectralNormConv2d(nn.Conv2d, SpectralNorm):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True,
                 num_svs=1, num_itrs=1, eps=1e-12):
        nn.Conv2d.__init__(self, in_channels, out_channels, kernel_size, stride,
                            padding, dilation, groups, bias)
        SpectralNorm.__init__(self, num_svs, num_itrs, out_channels, eps=eps)

    def forward(self, x):
        return F.conv2d(x, self.W_(), self.bias, self.stride,
                         self.padding, self.dilation, self.groups)


class SpectralNormLinear(nn.Linear, SpectralNorm):
    def __init__(self, in_features, out_features, bias=True,
                 num_svs=1, num_itrs=1, eps=1e-12):
        nn.Linear.__init__(self, in_features, out_features, bias)
        SpectralNorm.__init__(self, num_svs, num_itrs, out_features, eps=eps)

    def forward(self, x):
        return F.linear(x, self.W_(), self.bias)


class SpectralNormEmbedding(nn.Embedding, SpectralNorm):
    def __init__(self, num_embeddings, embedding_dim, padding_idx=None,
                 max_norm=None, norm_type=2, scale_grad_by_freq=False,
                 sparse=False, _weight=None,
                 num_svs=1, num_itrs=1, eps=1e-12):
        nn.Embedding.__init__(self, num_embeddings, embedding_dim, padding_idx,
                               max_norm, norm_type, scale_grad_by_freq,
                               sparse, _weight)
        SpectralNorm.__init__(self, num_svs, num_itrs, num_embeddings, eps=eps)

    def forward(self, x):
        return F.embedding(x, self.W_())

### Generatorの構築

![biggan_gen](images/biggan_gen.png)

BigGANは，Residual Networkをベースにネットワークが構築されています．先ほどまでに定義したレイヤやモジュールを用いてネットワークの構築をします．

図中のSplitは，潜在変数を任意の次元で分割する処理を表しています．Concatは，分割した潜在変数及びクラスラベルを結合する処理を表しています．図中には，単純にConv.と記載しましたが，実際にはSpectral Normalizationを施したConvolutionであることに注意してください．</cell id="cell-12">


In [ ]:
class GResBlock(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size=3, stride=1,
                 padding=1, n_cls=None, act='relu'):
        super().__init__()
        self.c1 = SpectralNormConv2d(in_channel, out_channel, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.c2 = SpectralNormConv2d(out_channel, out_channel, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.c1x1 = SpectralNormConv2d(in_channel, out_channel, kernel_size=1, stride=1, padding=0, bias=True)

        self.upsample_layer = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.cbn1 = ConditionalBatchNorm(in_channel, 128 + 20)
        self.cbn2 = ConditionalBatchNorm(out_channel, 128 + 20)

        if act == 'relu':
            self.activation = nn.ReLU()
        elif act == 'lrelu':
            self.activation = nn.LeakyReLU()
        elif act == 'tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError(f'{act} is not supported.')

    def forward(self, x, y):
        x1 = self.c1x1(self.upsample_layer(x))

        h1 = self.cbn1(x, y)
        h1 = self.activation(h1)
        h1 = self.upsample_layer(h1)
        h1 = self.c1(h1)

        h2 = self.cbn2(h1, y)
        h2 = self.activation(h2)
        h2 = self.c2(h2)

        return x1 + h2

In [ ]:
class Generator(nn.Module):
    def __init__(self, n_latent=120, n_ch=64, n_cls=100):
        super().__init__()
        self.n_ch = n_ch
        self.linear_latent = SpectralNormLinear(20, 4 * 4 * 16 * n_ch)
        self.linear_condition = SpectralNormLinear(n_cls, 128)

        self.blocks = nn.ModuleList([
            GResBlock(16 * n_ch, 16 * n_ch, n_cls=n_cls),
            GResBlock(16 * n_ch, 8 * n_ch, n_cls=n_cls),
            GResBlock(8 * n_ch, 4 * n_ch, n_cls=n_cls),
            GResBlock(4 * n_ch, 2 * n_ch, n_cls=n_cls),
            SelfAttention(2 * n_ch),
            GResBlock(2 * n_ch, 1 * n_ch, n_cls=n_cls),
        ])

        self.bn = nn.BatchNorm2d(1 * n_ch, eps=1e-4)
        self.relu = nn.ReLU()
        self.conv = SpectralNormConv2d(1 * n_ch, 3, kernel_size=3, stride=1, padding=1, bias=False)
        self.tanh = nn.Tanh()

        self.init_weights()

    def init_weights(self):
        self.param_count = 0
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, nn.Embedding)):
                init.orthogonal_(module.weight)
                self.param_count += sum(p.data.nelement() for p in module.parameters())
        print(f"Param count for G's initialized parameters: {self.param_count}")

    def forward(self, z, y):
        split_latent = torch.split(z, split_size_or_sections=20, dim=1)
        proj_cond = self.linear_condition(y)

        z_idx = 0
        out = self.linear_latent(split_latent[z_idx])
        out = out.view(-1, 16 * self.n_ch, 4, 4)
        for idx, block in enumerate(self.blocks):
            if idx == len(self.blocks) - 2:  # Self-Attention層はクラス条件を受け取らない
                out = block(out)
            else:
                z_idx += 1
                cond = torch.cat((split_latent[z_idx], proj_cond), dim=1)
                out = block(out, cond)

        out = self.bn(out)
        out = self.relu(out)
        out = self.conv(out)
        out = self.tanh(out)
        return out

### Discriminatorの構築

![biggan_dis](images/biggan_dis.png)

Discriminatorは，SN-GANやSA-GANと同じように構築します．また，cGANのように入力層に直接条件を与えるのではなく，Miyatoらが提案したprojection discriminatorのアイデアを用いて条件を与えます．

projection discriminatorは，条件を任意の次元のベクトル（Discriminatorの出力値と同じ次元数）へ埋め込み，画像から抽出した特徴との内積をDiscriminatorの出力値に足し合わせる，という簡単な方法で条件付けを行います．</cell id="cell-15">


In [ ]:
class DResBlock(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size=3, stride=1, padding=1, act='relu', down=True):
        super().__init__()
        self.c1x1 = SpectralNormConv2d(in_channel, out_channel, kernel_size=1, stride=1, padding=0, bias=True)
        self.c1 = SpectralNormConv2d(in_channel, out_channel, kernel_size=kernel_size, stride=stride, padding=padding, bias=True)
        self.c2 = SpectralNormConv2d(out_channel, out_channel, kernel_size=kernel_size, stride=stride, padding=padding, bias=True)

        if act == 'relu':
            self.activation = nn.ReLU()
        elif act == 'lrelu':
            self.activation = nn.LeakyReLU()
        elif act == 'tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError(f'{act} is not supported.')

        self.down = down
        if down:
            self.avg = nn.AvgPool2d(2, 2)

    def forward(self, x):
        if self.down:
            x1 = self.avg(self.c1x1(x))
        else:
            x1 = self.c1x1(x)

        h = self.c1(self.activation(x))
        h = self.c2(self.activation(h))

        if self.down:
            h = self.avg(h)

        return x1 + h

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, n_ch=64, n_cls=100):
        super().__init__()

        self.first_layer = nn.Sequential(
            SpectralNormConv2d(3, 1 * n_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(),
            SpectralNormConv2d(1 * n_ch, 1 * n_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.AvgPool2d(2, 2))

        self.first_skip = SpectralNormConv2d(3, 1 * n_ch, kernel_size=1, stride=1, padding=0, bias=True)
        self.avgpool = nn.AvgPool2d(2, 2)

        self.blocks = nn.Sequential(
            DResBlock(1 * n_ch, 2 * n_ch, down=True),
            SelfAttention(2 * n_ch),
            DResBlock(2 * n_ch, 4 * n_ch, down=True),
            DResBlock(4 * n_ch, 8 * n_ch, down=True),
            DResBlock(8 * n_ch, 16 * n_ch, down=True),
            DResBlock(16 * n_ch, 16 * n_ch, down=True),
            DResBlock(16 * n_ch, 16 * n_ch, down=False))
        self.relu = nn.ReLU()

        self.linear = SpectralNormLinear(16 * n_ch, 1)
        self.emb_cls = SpectralNormEmbedding(n_cls, 16 * n_ch)

        self.init_weights()

    def init_weights(self):
        self.param_count = 0
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, nn.Embedding)):
                init.orthogonal_(module.weight)
                self.param_count += sum(p.data.nelement() for p in module.parameters())
        print(f"Param count for D's initialized parameters: {self.param_count}")

    def forward(self, x, y):
        out = self.first_layer(x)
        out_skip = self.avgpool(self.first_skip(x))
        out = out + out_skip

        out = self.blocks(out)
        out = self.relu(out)
        out = out.view(out.size(0), out.size(1), -1)
        out = out.sum(dim=2)

        out_linear = self.linear(out).squeeze(1)
        embed = self.emb_cls(y)
        proj = (out * embed).sum(1)

        return out_linear + proj

### BigGANの学習
BigGANは非常に大規模なネットワークであり，演習時間内に学習を終えることが難しいため，学習を割愛し，学習済みモデルを用いて画像を生成します（学習のためのソースコードは，本ノートブックの後半に参考として掲載しています）．

今回ダウンロードするモデルは，LSUNというデータセットの中から10クラスのみを用いて学習したものです．画像サイズは128×128です．以下でpretrainモデルのzipファイルをダウンロードし，解凍します．中にはGeneratorのパラメータ**gen**と，Discriminatorのパラメータ**dis**が入っています．</cell id="cell-18">


In [ ]:
model_root = './biggan_model'
if not os.path.isdir(model_root):
    gdown.download(id='1j7a6cQoJoG3FZOz8wrG22Q7dLO-EArOP', output='biggan_model.zip', quiet=False)
    with zipfile.ZipFile('biggan_model.zip') as f:
        f.extractall('./')

### パラメータの定義

In [ ]:
categories = ['airplane', 'bird', 'bottle', 'bus', 'car',
              'cat', 'dog', 'horse', 'motorbike', 'sheep']

n_latent = 120
n_ch = 64
n_cls = len(categories)

G = Generator(n_latent, n_ch, n_cls).to(device)
D = Discriminator(n_ch, n_cls).to(device)
G.load_state_dict(torch.load('./biggan_model/gen', map_location=device))
D.load_state_dict(torch.load('./biggan_model/dis', map_location=device))

### BigGANを学習するためのソースコード
BigGANは非常に大規模なネットワークであり，本ノートブックが動作する環境では現実的な時間で学習を終えられません．そのため，以下のセルは実行せず，**参考用のソースコード**として掲載しています．学習リソースを用意できる方は，ご自身の環境で任意のデータセットを用いて学習してください（`training_dataset`は，本ノートブックには含まれていない，使用するデータセットに応じたDataLoaderです）．</cell id="cell-22">


In [ ]:
def ortho(model, strength=1e-4, blacklist=None):
    blacklist = blacklist or []
    with torch.no_grad():
        for param in model.parameters():
            # Only apply this to parameters with at least 2 axes, and not in the blacklist
            if len(param.shape) < 2 or any(param is item for item in blacklist):
                continue
            w = param.view(param.shape[0], -1)
            grad = (2 * torch.mm(torch.mm(w, w.t())
                    * (1. - torch.eye(w.shape[0], device=w.device)), w))
            param.grad.data += strength * grad.view(param.shape)

BigGANを学習する際の誤差関数は，hinge lossかwgan-gpの誤差関数を利用します．hinge lossの方が安定した学習で高解像な画像が生成される傾向にあるので，そちらを使用することをお勧めします．

In [ ]:
epochs = 100
beta1 = 0.0
beta2 = 0.999
lr_G, lr_D = 5e-5, 2e-4
G_ortho, D_ortho = 1e-4, 0.0
lambda_gp = 10
loss_func = 'hinge'
G_opt = optim.Adam(G.parameters(), lr=lr_G, betas=(beta1, beta2))
D_opt = optim.Adam(D.parameters(), lr=lr_D, betas=(beta1, beta2))

# training_dataset：(画像, クラスID) のペアを返すDataLoaderを，使用するデータセットに応じて別途用意してください
# tb：学習経過をログしたい場合は，torch.utils.tensorboard.SummaryWriterなどを別途用意してください

iteration = 0
for epoch in range(1, epochs + 1):
    print(f'Training networks for epoch {epoch}.')
    for idx, (img, tgt) in enumerate(training_dataset):
        G.train()
        D.train()
        real_img = img.to(device)
        tgt = tgt.to(device)
        onehot = torch.eye(n_cls, device=device)[tgt].type_as(real_img)

        z = torch.randn(real_img.size(0), n_latent).to(device)  # DataLoaderが返す実際のバッチサイズ（最後のバッチは小さくなりうる）に合わせる
        fake_img = G(z, onehot)

        # ====================== Update Discriminator ======================
        D.zero_grad()
        dis_real_out = D(real_img, tgt)
        dis_fake_out = D(fake_img.detach(), tgt)  # Discriminatorの更新にはGeneratorの勾配は不要なのでdetachする

        if loss_func == 'hinge':
            dis_real = F.relu(1.0 - dis_real_out).mean()
            dis_fake = F.relu(1.0 + dis_fake_out).mean()
            dis_loss = dis_real + dis_fake
        elif loss_func == 'wgan-gp':
            dis_loss = -dis_real_out.mean() + dis_fake_out.mean()

            # Gradient Penalty（詳細は`wgan_gp.ipynb`を参照）
            eps = torch.rand(real_img.size(0), 1, 1, 1, device=device).expand_as(real_img)
            interpolated = (eps * real_img + (1 - eps) * fake_img.detach()).requires_grad_(True)
            out = D(interpolated, tgt)
            grad = torch.autograd.grad(outputs=out, inputs=interpolated,
                                       grad_outputs=torch.ones(out.size(), device=device),
                                       retain_graph=True, create_graph=True, only_inputs=True)[0]
            grad = grad.view(grad.size(0), -1)
            grad_l2norm = torch.sqrt(torch.sum(grad ** 2, dim=1))
            d_loss_gp = torch.mean((grad_l2norm - 1) ** 2)

            dis_loss = dis_loss + lambda_gp * d_loss_gp

        dis_loss.backward()

        if D_ortho > 0.0:
            ortho(D, D_ortho)

        D_opt.step()

        # ====================== Update Generator ======================
        G.zero_grad()
        z = torch.randn(real_img.size(0), n_latent).to(device)
        fake_img = G(z, onehot)
        gen_out = D(fake_img, tgt)
        gen_loss = -gen_out.mean()
        gen_loss.backward()

        if G_ortho > 0.0:
            ortho(G, G_ortho)

        G_opt.step()

        iteration += 1
        if idx % 100 == 0:
            print(f'Training epoch: {epoch} [{idx * len(img)}/{len(training_dataset.dataset)} '
                  f'({100. * idx / len(training_dataset):.0f}%)] | D loss: {dis_loss.item():.6f} | G loss: {gen_loss.item():.6f} |')
            # tb.add_scalars('prediction loss', {'D': dis_loss.item(), 'G': gen_loss.item()}, iteration)

### 画像の生成
実際に画像を生成してみましょう．

生成するクラスは，任意に設定することができます．
特に何も指定をしなければランダムに決定したクラスを生成することになります．
自分の好きなクラスを生成したいときは，**cls_idx**を適当なintの数字にしてください．

潜在変数`z`の値の絶対値が大きい（分布の裾にある）ほど，見た目が破綻した画像が生成されやすくなります．そこで，閾値`threshold`を超える要素を正規分布から再サンプリングし直すことで，多様性と引き換えに生成画像の破綻を抑える**Truncation Trick**という手法を実装します．

In [ ]:
def resampling_module(org_noise, threshold=2):
    """閾値の絶対値を超える要素を，正規分布から再サンプリングした値に置き換える（Truncation Trick）"""
    noise = org_noise.clone()
    mask = noise.abs() > threshold
    while mask.any():
        noise[mask] = torch.randn(int(mask.sum().item()), device=noise.device)
        mask = noise.abs() > threshold
    return noise

In [ ]:
cls_idx = None
use_resampling = False
threshold = 1
n_img = 10

if cls_idx is None:  # cls_idx=0（先頭のクラス）を指定した場合でも正しく判定されるよう，Noneかどうかで判定する
    cls_idx = np.random.randint(n_cls)

latent = torch.randn(n_img, n_latent).to(device)
onehot = torch.eye(n_cls)[cls_idx].unsqueeze(0).expand(n_img, n_cls).to(device)
if use_resampling:
    latent = resampling_module(latent, threshold=threshold)

G.eval()
with torch.no_grad():
    imgs = G(latent, onehot)

print(f'Generated class: {categories[cls_idx]}')

row = 2
col = 5
plt.figure(figsize=(5, 2))

num = 0
while num < row * col:
    img = (imgs[num] * 256.).permute(1, 2, 0).clamp(min=0., max=255.).cpu().numpy().astype(np.uint8)
    num += 1
    plt.subplot(row, col, num)
    plt.imshow(img)
    plt.axis('off')

## 課題

1. `resampling_module`の閾値（`threshold`）を変更したらどのようになっていくか確認してみましょう．
2. 任意の2つの潜在変数の間を線形補間した場合に，どのような画像が生成されるか確認してみましょう．
3. `cls_idx`を変えて異なるクラスの画像を生成し，Class-conditional Batch Normalizationとprojection discriminatorによる条件付けが機能していることを確認してください．</cell id="cell-30">
